# Acoustic & Vibration Fault Diagnosis - Custom Deep Learning & Kaggle GPU Pipeline

This notebook trains and benchmarks end-to-end Deep Learning and Machine Learning models on the **12-Class Acoustic Fault Dataset** (`Base_de_Dados`):
- **Custom Time-Frequency ResNet-Attention Model (`TF-FaultNet`)**: Differentiable GPU STFT/Log-Mel Spectrogram extraction + Multi-Scale Residual CNN with Squeeze-and-Excitation Attention.
- **1D Waveform CNN**: Raw audio feature extraction baseline.
- **Bidirectional GRU**: Recurrent temporal sequence baseline on time-frequency frames.
- **XGBoost Classifier**: Multi-class gradient boosting on spectral/temporal statistical indicators.

### Dataset Classes (12 Balanced Fault Modes):
1. `Normal`: Normal operational regime
2. `Escorregamento`, `Escorregamento_P1`, `Escorregamento_P1P4`: Belt slipping across configurations
3. `Perda_concentrada`, `Perda_concentrada_P1`, `Perda_concentrada_P1P4`: Concentrated tooth/element loss
4. `Perda_material`, `Perda_material_P1`, `Perda_material_P1P4`: Surface wear and material degradation
5. `Sem_P1`, `Sem_P1P4`: Structural/pulley missing configurations


## 1. Environment & GPU Setup

In [ ]:
import os, sys, glob, wave, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
import xgboost as xgb

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## 2. Locate Dataset

In [ ]:
# Search paths for Kaggle or Local workspace
search_roots = ["/kaggle/input", "./Base_de_Dados", "../Base_de_Dados", "./"]
data_dir = None
for root in search_roots:
    hits = glob.glob(os.path.join(root, "**/Normal"), recursive=True)
    if hits:
        data_dir = os.path.dirname(hits[0])
        break

if not data_dir or not os.path.isdir(data_dir):
    raise FileNotFoundError("Base_de_Dados dataset directory not found!")

print(f"Dataset located at: {data_dir}")
class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
print(f"Found {len(class_names)} classes: {class_names}")


## 3. Signal Processing & Dataset Loader

In [ ]:
def load_wav(filepath):
    with wave.open(filepath, 'rb') as w:
        n_channels = w.getnchannels()
        sampwidth = w.getsampwidth()
        framerate = w.getframerate()
        n_frames = w.getnframes()
        frames = w.readframes(n_frames)
        
    dtype = np.int16 if sampwidth == 2 else (np.int32 if sampwidth == 4 else np.uint8)
    audio = np.frombuffer(frames, dtype=dtype).astype(np.float32)
    if sampwidth == 2:
        audio /= 32768.0
    elif sampwidth == 4:
        audio /= 2147483648.0
    elif sampwidth == 1:
        audio = (audio - 128.0) / 128.0
        
    if n_channels > 1:
        audio = audio.reshape(-1, n_channels).mean(axis=1)
    return audio, framerate

class FaultAudioDataset(Dataset):
    def __init__(self, filepaths, labels, target_len=44100, augment=False):
        self.filepaths = filepaths
        self.labels = labels
        self.target_len = target_len
        self.augment = augment

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        path = self.filepaths[idx]
        label = self.labels[idx]
        audio, sr = load_wav(path)
        
        if len(audio) < self.target_len:
            audio = np.pad(audio, (0, self.target_len - len(audio)), mode='constant')
        else:
            audio = audio[:self.target_len]
            
        if self.augment:
            if np.random.rand() < 0.4:
                audio += np.random.normal(0, 0.005, size=audio.shape)
            if np.random.rand() < 0.4:
                audio *= np.random.uniform(0.8, 1.2)
                
        return torch.from_numpy(audio).float(), torch.tensor(label, dtype=torch.long)


## 4. Differentiable GPU Mel-Spectrogram & TF-FaultNet Architecture

In [ ]:
def hz_to_mel(hz):
    return 2595.0 * np.log10(1.0 + hz / 700.0)

def mel_to_hz(mel):
    return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)

def get_mel_filterbank(sr=44100, n_fft=1024, n_mels=64, f_min=20.0, f_max=22050.0):
    mel_min = hz_to_mel(f_min)
    mel_max = hz_to_mel(f_max)
    mel_points = np.linspace(mel_min, mel_max, n_mels + 2)
    hz_points = mel_to_hz(mel_points)
    bin_points = np.floor((n_fft + 1) * hz_points / sr).astype(int)
    n_freqs = n_fft // 2 + 1
    weights = np.zeros((n_mels, n_freqs), dtype=np.float32)
    for i in range(1, n_mels + 1):
        left, center, right = bin_points[i-1], bin_points[i], bin_points[i+1]
        for f in range(left, center):
            if f < n_freqs and center > left: weights[i-1, f] = (f - left) / (center - left)
        for f in range(center, right):
            if f < n_freqs and right > center: weights[i-1, f] = (right - f) / (right - center)
    enorm = 2.0 / (hz_points[2:n_mels+2] - hz_points[:n_mels])
    weights *= enorm[:, np.newaxis]
    return torch.from_numpy(weights).float()

class LogMelSpectrogramExtractor(nn.Module):
    def __init__(self, sr=44100, n_fft=1024, hop_length=256, n_mels=64):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.register_buffer("window", torch.hann_window(n_fft))
        self.register_buffer("mel_fb", get_mel_filterbank(sr, n_fft, n_mels))
        
    def forward(self, x):
        stft = torch.stft(x, n_fft=self.n_fft, hop_length=self.hop_length, win_length=self.n_fft,
                          window=self.window, return_complex=True, center=True, pad_mode='reflect')
        power = torch.abs(stft) ** 2
        mel = torch.matmul(self.mel_fb, power)
        log_mel = torch.log(torch.clamp(mel, min=1e-6))
        return log_mel.unsqueeze(1)

class SqueezeExcitation(nn.Module):
    def __init__(self, channels, r=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, max(1, channels // r), bias=False), nn.ReLU(inplace=True),
            nn.Linear(max(1, channels // r), channels, bias=False), nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.fc(x).view(x.size(0), x.size(1), 1, 1)

class ConvBlock2D(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.se = SqueezeExcitation(out_c)
        self.shortcut = nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False), nn.BatchNorm2d(out_c)) if (stride != 1 or in_c != out_c) else nn.Identity()
    def forward(self, x):
        res = self.shortcut(x)
        out = self.se(self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x)), inplace=True))))
        return F.relu(out + res, inplace=True)

class TFFaultNet(nn.Module):
    def __init__(self, num_classes=12):
        super().__init__()
        self.spec_extractor = LogMelSpectrogramExtractor()
        self.init_conv = nn.Sequential(
            nn.Conv2d(1, 32, 5, stride=2, padding=2, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2)
        )
        self.layer1 = ConvBlock2D(32, 64, stride=2)
        self.layer2 = ConvBlock2D(64, 128, stride=2)
        self.layer3 = ConvBlock2D(128, 256, stride=2)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.max_pool = nn.AdaptiveMaxPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(0.35), nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.2), nn.Linear(256, num_classes)
        )
    def forward(self, x):
        spec = self.spec_extractor(x)
        feat = self.layer3(self.layer2(self.layer1(self.init_conv(spec))))
        pooled = torch.cat([self.avg_pool(feat).view(x.size(0), -1), self.max_pool(feat).view(x.size(0), -1)], dim=1)
        return self.classifier(pooled)


## 5. Model Training & 5-Fold Cross Validation

In [ ]:
# Gather all files
filepaths, labels = [], []
class_to_idx = {c: i for i, c in enumerate(class_names)}
for cls in class_names:
    for w in sorted(glob.glob(os.path.join(data_dir, cls, "*.wav"))):
        filepaths.append(w)
        labels.append(class_to_idx[cls])
filepaths, labels = np.array(filepaths), np.array(labels)
print(f"Loaded {len(filepaths)} files across {len(class_names)} classes.")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_preds, all_targets = [], []
fold_accs, fold_f1s = [], []

for fold, (train_idx, val_idx) in enumerate(skf.split(filepaths, labels), 1):
    train_ds = FaultAudioDataset(filepaths[train_idx], labels[train_idx], augment=True)
    val_ds = FaultAudioDataset(filepaths[val_idx], labels[val_idx], augment=False)
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
    
    model = TFFaultNet(num_classes=len(class_names)).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))
    
    best_f1, best_preds, best_targets = 0.0, None, None
    for ep in range(15):
        model.train()
        for aud, lbl in train_loader:
            aud, lbl = aud.to(device), lbl.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                loss = criterion(model(aud), lbl)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
        model.eval()
        p_list, t_list = [], []
        with torch.no_grad():
            for aud, lbl in val_loader:
                aud = aud.to(device)
                with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                    out = model(aud)
                p_list.extend(torch.argmax(out, dim=1).cpu().numpy())
                t_list.extend(lbl.numpy())
                
        v_acc = accuracy_score(t_list, p_list)
        v_f1 = f1_score(t_list, p_list, average='macro')
        if v_f1 > best_f1:
            best_f1 = v_f1
            best_preds = p_list
            best_targets = t_list
            
    fold_acc = accuracy_score(best_targets, best_preds)
    fold_accs.append(fold_acc)
    fold_f1s.append(best_f1)
    all_preds.extend(best_preds)
    all_targets.extend(best_targets)
    print(f"Fold {fold}/5 -> Accuracy: {fold_acc*100:.2f}% | Macro F1: {best_f1*100:.2f}%")

print("=" * 60)
print(f"Overall 5-Fold Mean Accuracy: {np.mean(fold_accs)*100:.2f}% ± {np.std(fold_accs)*100:.2f}%")
print(f"Overall 5-Fold Mean Macro F1: {np.mean(fold_f1s)*100:.2f}% ± {np.std(fold_f1s)*100:.2f}%")


## 6. Evaluation Visualizations & Confusion Matrix

In [ ]:
cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(10, 8), dpi=150)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('TF-FaultNet Confusion Matrix (5-Fold Stratified CV)', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Predicted Label', fontweight='bold')
plt.ylabel('True Label', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

prec, rec, f1, supp = precision_recall_fscore_support(all_targets, all_preds)
report_df = pd.DataFrame({
    'Class': class_names,
    'Precision': [f'{p*100:.2f}%' for p in prec],
    'Recall': [f'{r*100:.2f}%' for r in rec],
    'F1-Score': [f'{f*100:.2f}%' for f in f1],
    'Support': supp
})
display(report_df)
